# cc-site-extract.ipynb

extracts course data from https://commoncore.hku.hk.  
outputs as JSON.  

## fields
- course name
- sustainable development goals
- semester offered
- study load distribution (and total study load)
- assignment distribution

In [6]:
# extract course list from https://commoncore.hku.hk

import requests
from bs4 import BeautifulSoup

courses = set()
course_names = {}

def scan_inquiry(inquiry):
	url = f"https://commoncore.hku.hk/{inquiry}/"
	print(f"fetching {url}...")

	response = requests.get(url)
	if response.status_code != 200:
		raise RuntimeError("bad response")
	
	soup = BeautifulSoup(response.text, "html.parser")

	for elem in soup.select(".elementor-widget-container:has(> h5)"):
		h5 = elem.find("h5")
		course_code = h5.get_text().strip()
		course_name = h5.find_next_sibling().get_text()
		
		print(f"{course_code}: {course_name}")
		courses.add(course_code)
		course_names[course_code] = course_name

inquiries = [
	"scientific-and-technological-literacy",
	"arts-and-humanities",
	"global-issues",
	"china-culture-state-and-society",
	"artificial-intelligence",
]
for inquiry in inquiries:
	scan_inquiry(inquiry)

print(f"len={len(courses)}")


fetching https://commoncore.hku.hk/scientific-and-technological-literacy/...
CCST5035:  Making Sense of Science-related Social Issues (Online) 
CCST5037: Mathematics: A Cultural Heritage (Online)
CCST9003: Everyday Computing and the Internet
CCST9006: Chasing Biomedical Miracles: Promises and Perils
CCST9009: Living with Stem Cells
CCST9012: Our Place in the Universe
CCST9013: Our Living Environment
CCST9014: Science and Music
CCST9016: Energy: Its Evolution and Environmental Impacts
CCST9017: Hidden Order in Daily Life: A Mathematical Perspective
CCST9018: Origin and Evolution of Life
CCST9020: Sustainable Development of the Built Environment
CCST9022: How the Mass Media Depicts Science, Technology and the Natural World
CCST9023: The Oceans: Science and Society
CCST9025: Genetics and Human Nature
CCST9026: Scientific Revolutions: Their Continuing Impact on Our World and Society
CCST9027: The Science of Irrational Thinking
CCST9030: Forensic Science: Unmasking Evidence, Mysteries and C

In [ ]:
# if sis-info.json already exists, search those courses as well

import os.path
import json

sis_path = "sis-info.json"

if os.path.isfile(sis_path):
	print(f"extracting from {sis_path}...")
	with open(sis_path, "r") as file:
		data = json.load(file)
		courses.update(map(lambda x : x.strip(), data.keys()))

print(f"len={len(courses)}")
	


extracting from sis-info.json...
len=364


In [12]:
# traverse course list and get course info per course

import concurrent.futures
import threading
import time

course_info = {}
course_info_lock = threading.Lock()

def scan_course(course):
	url = f"https://commoncore.hku.hk/{course}/"

	for _ in range(3):
		print(f"fetching {url}...")
		response = requests.get(url)
		if response.status_code == 200 or response.status_code == 404:
			break
		print("fail, retry...")
		time.sleep(1)
	if response.status_code == 200:
		pass
	elif response.status_code == 404:
		print(f"{course} not found :(")
		return
	else:
		raise RuntimeError("bad response (%d)" % response.status_code)
	
	soup = BeautifulSoup(response.text, "html.parser")

	# sustainable development goals
	sdgs = [ elem["alt"] for elem in soup.select(".span6.zmt.sdgs-item > img") ]
	# print(sdgs)

	# semester offered
	sem_offered = soup.find(id="osdt").find_next_sibling().get_text().strip()
	# print(sem_offered)

	# study load
	rows = soup.select_one("#load + table").select("tr")
	study_load_dist = {}
	for elem in rows[1:-1]:
		activity, hours = elem.find_all("td")
		study_load_dist[activity.get_text()] = hours.get_text().strip()
	study_load_total = rows[-1].find_all("td")[-1].get_text().strip()
	# print(study_load_dist, study_load_total)

	# assessment
	rows = soup.select_one("#ass + table").select("tr")
	ass_dist = {}
	for elem in rows[1:]:
		task, weight = elem.find_all("td")
		ass_dist[task.get_text().strip()] = weight.get_text().strip()
	# print(ass_dist)

	with course_info_lock:
		course_info[course] = {
			"course_name": course_names[course],
			"sdgs": sdgs,
			"sem_offered": sem_offered,
			"study_load_dist": study_load_dist,
			"study_load_total": study_load_total,
			"ass_dist": ass_dist,
		}

with concurrent.futures.ThreadPoolExecutor() as executor:
	executor.map(scan_course, courses)

print("done!")


fetching https://commoncore.hku.hk/CCHU9005/...
fetching https://commoncore.hku.hk/CCHU9065/...
fetching https://commoncore.hku.hk/CCST9005/...
fetching https://commoncore.hku.hk/CCAI9021/...
fetching https://commoncore.hku.hk/CCGL9007/...
fetching https://commoncore.hku.hk/CCHU9093/...
fetching https://commoncore.hku.hk/CCGL9056/...
CCST9005 not found :(
fetching https://commoncore.hku.hk/CCGL9060/...
fetching https://commoncore.hku.hk/CCCH9018/...
fetching https://commoncore.hku.hk/CCHU5065/...
fetching https://commoncore.hku.hk/CCCH9024/...
fetching https://commoncore.hku.hk/CCHU9010/...
CCGL9060 not found :(
fetching https://commoncore.hku.hk/CCGL9025/...
fetching https://commoncore.hku.hk/CCGL5062/...
fetching https://commoncore.hku.hk/CCCH9008/...
fetching https://commoncore.hku.hk/CCHU9009/...
fetching https://commoncore.hku.hk/CCST9015/...
fetching https://commoncore.hku.hk/CCHU5088/...
fetching https://commoncore.hku.hk/CCAI9032/...
fetching https://commoncore.hku.hk/CCGL9083/

In [ ]:
import json

output_path = "cc-site-info.json"

print(json.dumps(course_info, indent=4))

with open(output_path, "w") as file:
	json.dump(course_info, file, indent=4)

200
